# RF-DETR Video Inference -- Marine Object Detection + Tracking

Runs a fine-tuned RF-DETR-Small checkpoint over a video, tracks detections with ByteTrack (via `supervision`), and applies an identity-smoothing / re-link / lock / coast layer so each tracked object keeps a stable class label and identity across the clip.

**Before running:**
1. Set `MODEL_PATH` to your RF-DETR checkpoint (the EMA checkpoint from fine-tuning, e.g. `checkpoint_best_ema.pth`).
2. Set `CLASS_NAMES` to match your `data.yaml` class order exactly -- RF-DETR checkpoints don't carry class names the way Ultralytics `.pt` files do.
3. Set `DEVICE = "cuda"` if you have a GPU available.


In [ ]:
import cv2
import numpy as np
import torch
import supervision as sv
from collections import defaultdict
from rfdetr import RFDETRSmall

# --- Configuration ---
MODEL_PATH = "weights/checkpoint_best_ema.pth"
VIDEO_PATH = "samples/input.mp4"
OUTPUT_PATH = "out/output_annotated.mp4"

# RF-DETR doesn't expose class names on the checkpoint the way Ultralytics does (model.names) --
# this must match the exact class order used during training (CLASS_NAMES from data.yaml).
CLASS_NAMES = ['boat', 'buoy', 'cargo', 'ferry', 'tug', 'warship', 'yacht']

CONF_THRESH = 0.40     # detection confidence threshold
NMS_IOU_THRESH = 0.30  # RF-DETR is NMS-free by design, but class-agnostic NMS is still applied
                        # once after prediction (parity with agnostic_nms=True in YOLO pipelines)
IMGSZ = 512             # RF-DETR-Small's native training resolution
DEVICE = "cpu"          # set to "cuda" if you have a CUDA GPU available

# --- Smoothing / identity config ---
MIN_HITS_TO_DISPLAY = 5     # don't show a label until an identity has this many detections
SWITCH_MARGIN = 2.0         # new class's cumulative score must beat current by this factor to switch (only while unlocked)
DECAY = 0.98                # per-frame decay on old evidence (long memory, adapts very slowly)
LOCK_AFTER_HITS = 20        # once an identity has this many confirmed detections, PERMANENTLY freeze its class.
                             # Why: a real sustained model error (e.g. same ship called "tug" for 70+ straight
                             # frames due to a scale/angle change) will eventually out-vote any reactive smoothing
                             # no matter how decayed. Locking after enough confirmed hits trades "can self-correct
                             # forever" for "stays consistent for an object that's clearly already identified" --
                             # which is what you actually want for a single object tracked continuously in-frame.

# --- Size-growth unlock: break a permanent lock if the object's box grows a lot ---
# Rationale: a lock can fire based on a partial/occluded/distant view (e.g. only the bow of a
# cargo ship poking into frame gets misread as "tug"). If the box later grows substantially --
# meaning we're now seeing much more of the object, or it's much closer -- that earlier lock was
# based on incomplete evidence, so we deliberately re-open classification rather than staying wrong
# forever. A short re-confirmation window (RECLASSIFY_MIN_HITS) then prevents a single noisy frame
# right after unlocking from immediately flipping the label again.
SIZE_GROWTH_UNLOCK_FACTOR = 1.8   # box area must grow to >= this multiple of the area at lock time to trigger re-evaluation
RECLASSIFY_MIN_HITS = 5           # hits required after an unlock before a class switch is allowed again

# --- Re-link config (handles ByteTrack assigning a *new* ID to the same object) ---
RELINK_MAX_FRAMES_GONE = 45   # how many frames a track can be "lost" and still be re-linked
RELINK_MAX_CENTER_DIST = 80   # max center-point movement (px) to consider it the same object
RELINK_MAX_SIZE_RATIO = 1.6   # box area can't change by more than this factor between the two

# --- Box coasting (visual only): hold the last known box for a few frames if
# the detector/tracker drops the object for a moment, so it doesn't flicker
# in and out. Purely cosmetic -- does not affect classification evidence.
COAST_MAX_FRAMES = 8

# --- ByteTrack, via supervision, standing in for Ultralytics' built-in model.track() ---
# (RF-DETR has no native .track() / tracker-config file the way Ultralytics does -- supervision's
# ByteTrack operates directly on the sv.Detections that RF-DETR's .predict() already returns, and
# exposes the same knobs a custom bytetrack_stable.yaml would set: track_buffer=60 (doubled from
# the default 30, to survive brief occlusions/misses) and match_thresh=0.85.)
tracker = sv.ByteTrack(
    track_activation_threshold=0.25,      # == track_high_thresh
    lost_track_buffer=60,                 # == track_buffer
    minimum_matching_threshold=0.85,      # == match_thresh
    frame_rate=30,
)

# Load the fine-tuned model
model = RFDETRSmall(pretrain_weights=MODEL_PATH)
model.model.model.eval()
if DEVICE == "cuda":
    model.model.model.cuda()

# --- Video Reader / Writer Setup ---
cap = cv2.VideoCapture(VIDEO_PATH)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS)) or 30
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))
tracker.frame_rate = fps  # keep ByteTrack's internal max_time_lost consistent with actual video fps


class Identity:
    """A persistent object identity. May outlive multiple raw tracker IDs
    if the tracker drops and reassigns a new ID for the same real object."""
    __slots__ = ("cls_scores", "committed_cls", "hits", "last_box",
                 "last_frame_seen", "raw_track_ids", "locked",
                 "area_at_lock", "hits_since_reset")

    def __init__(self):
        self.cls_scores = defaultdict(float)
        self.committed_cls = None
        self.hits = 0
        self.last_box = None          # (x1, y1, x2, y2)
        self.last_frame_seen = -1
        self.raw_track_ids = set()
        self.locked = False           # once True, committed_cls is frozen -- unless broken by a size-growth unlock
        self.area_at_lock = None      # box_area at the moment `locked` was set True
        self.hits_since_reset = 0     # hits since track start / since the last size-growth unlock

    def update(self, cls_idx, conf, box, frame_idx):
        self.hits += 1
        self.last_box = box
        self.last_frame_seen = frame_idx

        if self.locked:
            # Even though the class is normally frozen once locked, a large jump in box size
            # means we're now seeing much more of the object than we were when it locked in
            # (e.g. only a corner of a cargo ship's bow was visible, and it has since come
            # fully into frame or much closer to the camera). That earlier lock was based on
            # incomplete evidence, so break it and let the identity re-classify from scratch.
            area = box_area(box)
            if self.area_at_lock is not None and area >= SIZE_GROWTH_UNLOCK_FACTOR * self.area_at_lock:
                self.locked = False
                self.cls_scores.clear()
                self.hits_since_reset = 0
                self.area_at_lock = None
            else:
                return  # class already finalized -- ignore new evidence, keep coasting the box

        self.hits_since_reset += 1

        # Decay old evidence slightly before adding new evidence, so an
        # identity's classification is dominated by its long-run history
        # rather than the most recent few frames -- this is what stops a
        # correctly-classified "cargo" from drifting to "tug" after a
        # stretch of ambiguous frames.
        for k in self.cls_scores:
            self.cls_scores[k] *= DECAY
        self.cls_scores[cls_idx] += conf

        best_cls = max(self.cls_scores, key=self.cls_scores.get)
        if self.committed_cls is None:
            if self.hits >= MIN_HITS_TO_DISPLAY:
                self.committed_cls = best_cls
        elif best_cls != self.committed_cls:
            # After a size-growth unlock, require a short re-confirmation window
            # (RECLASSIFY_MIN_HITS) before allowing a switch, so a single noisy frame
            # right after the object grows doesn't immediately flip the label again.
            if self.hits_since_reset >= RECLASSIFY_MIN_HITS and \
                    self.cls_scores[best_cls] >= SWITCH_MARGIN * self.cls_scores.get(self.committed_cls, 1e-6):
                self.committed_cls = best_cls

        # Once we've seen enough confirmed detections since track start (or since the last
        # size-growth unlock), permanently lock the class in. This is what actually guarantees
        # consistency for an object that stays in frame -- no amount of later disagreement (even
        # a long, confident stretch) can flip it after this point, unless the box grows a lot.
        if self.committed_cls is not None and self.hits_since_reset >= LOCK_AFTER_HITS:
            self.locked = True
            self.area_at_lock = box_area(box)


def box_center(b):
    x1, y1, x2, y2 = b
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)


def box_area(b):
    x1, y1, x2, y2 = b
    return max(0, x2 - x1) * max(0, y2 - y1)


# raw ByteTrack id -> Identity
identities = {}
# identities that stopped receiving updates recently (candidates for re-linking)
lost_identities = []  # list of Identity objects


In [ ]:
print(f"Processing video. Output will be saved to: {OUTPUT_PATH}")

frame_idx = 0
seen_raw_ids_this_frame = set()

while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break
    frame_idx += 1
    seen_raw_ids_this_frame.clear()
    frame = frame_bgr.copy()

    # --- RF-DETR inference (equivalent of the YOLO model.track(..., stream=True) call) ---
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    detections = model.predict(frame_rgb, threshold=CONF_THRESH)

    # class-agnostic NMS, equivalent to YOLO's agnostic_nms=True -- stops overlapping
    # predictions of different classes from both being drawn/tracked
    if len(detections) > 0:
        detections = detections.with_nms(threshold=NMS_IOU_THRESH, class_agnostic=True)

    # --- ByteTrack, equivalent of Ultralytics' built-in tracker + TRACKER_CFG_PATH ---
    detections = tracker.update_with_detections(detections)

    if detections.tracker_id is not None and len(detections) > 0:
        boxes = detections.xyxy.astype(int)
        raw_ids = detections.tracker_id.astype(int)
        clss = detections.class_id.astype(int)
        confs = detections.confidence

        for box, raw_id, cls_idx, conf in zip(boxes, raw_ids, clss, confs):
            seen_raw_ids_this_frame.add(raw_id)

            identity = identities.get(raw_id)

            if identity is None:
                # New raw track ID from the tracker. Before treating it as a
                # brand-new object, check whether it's actually a recently
                # lost identity reappearing under a new ID (this is the main
                # fix for "cargo becomes tug after a while").
                best_match, best_dist = None, None
                for cand in lost_identities:
                    if frame_idx - cand.last_frame_seen > RELINK_MAX_FRAMES_GONE:
                        continue
                    if cand.last_box is None:
                        continue
                    dist = np.hypot(*(np.array(box_center(box)) - np.array(box_center(cand.last_box))))
                    if dist > RELINK_MAX_CENTER_DIST:
                        continue
                    area_ratio = max(box_area(box), 1) / max(box_area(cand.last_box), 1)
                    if not (1 / RELINK_MAX_SIZE_RATIO <= area_ratio <= RELINK_MAX_SIZE_RATIO):
                        continue
                    if best_dist is None or dist < best_dist:
                        best_match, best_dist = cand, dist

                if best_match is not None:
                    identity = best_match
                    lost_identities.remove(best_match)
                else:
                    identity = Identity()

                identity.raw_track_ids.add(raw_id)
                identities[raw_id] = identity

            identity.update(cls_idx, float(conf), tuple(box), frame_idx)

            if identity.committed_cls is None:
                continue  # not enough evidence yet -- don't draw a label

            class_name = CLASS_NAMES[identity.committed_cls]
            label_text = f"{class_name} {conf:.2f}"
            x1, y1, x2, y2 = box
            color = (0, 200, 255)
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
            cv2.putText(frame, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)

    # Any raw ID that was active before but missing this frame becomes a
    # "lost" candidate for re-linking, and (briefly) coasts its last box so
    # the annotation doesn't flicker off for a couple of dropped frames.
    for raw_id, identity in list(identities.items()):
        if raw_id in seen_raw_ids_this_frame:
            continue
        gone_for = frame_idx - identity.last_frame_seen
        if gone_for == 1:
            lost_identities.append(identity)
        if gone_for <= COAST_MAX_FRAMES and identity.committed_cls is not None:
            class_name = CLASS_NAMES[identity.committed_cls]
            x1, y1, x2, y2 = identity.last_box
            color = (0, 150, 255)  # slightly different shade to indicate "coasting"
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            label_text = f"{class_name} (coasting)"
            (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(frame, (x1, y1 - text_h - 5), (x1 + text_w, y1), color, -1)
            cv2.putText(frame, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
        elif gone_for > RELINK_MAX_FRAMES_GONE:
            # fully expire -- stop tracking/re-linking this identity
            identities.pop(raw_id, None)
            if identity in lost_identities:
                lost_identities.remove(identity)

    out.write(frame)

# Clean up
cap.release()
out.release()
print("Inference and rendering complete!")
